# Phase 2: Progressive ExPSO Compression + Adversarial Fine-Tuning
**Algorithm 3:** Progressive compression at 30%, 50%, 70% with BC-ExPSO optimization.
Creates 3 diverse expert models for SME ensemble (Phase 3) and MVC-NNT (Phase 4).

### Fixes applied:
- Cumulative loss: batch-loop accumulation (Eq. 1), removed incorrect gbest anchoring
- Velocity damping: sigmoid-based nonlinear damping (Algorithm 2)
- Exponential factor: per-layer norm (Algorithm 1)
- Progressive compression: iterative pruning (Algorithm 3)

In [1]:
import os, sys, gc, time, warnings
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchattacks
import numpy as np
warnings.filterwarnings('ignore')

from utils import (
    device, get_cifar10_loaders, evaluate, train_one_epoch,
    save_checkpoint, load_checkpoint, count_zero_params
)
from expsoptimizer import ExPSOPruner, progressive_prune

print(f'Device: {device}')

Device: cuda


In [2]:
trainloader, testloader = get_cifar10_loaders()
print('Data loaded successfully.')

Data loaded successfully.


In [3]:
# ==========================================================================
# CONFIGURATION
# ==========================================================================
TRAIN_EPOCHS = 80          # Adversarial fine-tuning epochs per expert
PROGRESSIVE_STEPS = 3      # Algorithm 3: number of progressive stages
EXPSO_PARTICLES = 30       # Swarm size
EXPSO_ITER_PER_STEP = 30   # ExPSO iterations per progressive step
INTER_STEP_FT_EPOCHS = 5   # Brief fine-tune between progressive steps
BASE_LR = 0.05             # Learning rate for adversarial fine-tuning

eps, alpha, steps = 8/255, 2/255, 20

print(f'Progressive Steps: {PROGRESSIVE_STEPS}')
print(f'ExPSO Particles: {EXPSO_PARTICLES}')
print(f'ExPSO Iterations/Step: {EXPSO_ITER_PER_STEP}')
print(f'Fine-tune Epochs: {TRAIN_EPOCHS}')

Progressive Steps: 3
ExPSO Particles: 30
ExPSO Iterations/Step: 30
Fine-tune Epochs: 80


In [4]:
# Load and evaluate base model
base_model = torchvision.models.resnet18(weights=None, num_classes=10).to(device)
base_model = load_checkpoint(base_model, 'checkpoints/resnet18_standard_best.pth').float()

clean_base = evaluate(base_model, testloader)
robust_base = evaluate(base_model, testloader,
    atk=torchattacks.PGD(base_model, eps=eps, alpha=alpha, steps=steps))

print(f'Base Model: Clean={clean_base:.2f}%, Robust(PGD-20)={robust_base:.2f}%')
print(f'Sparsity: {count_zero_params(base_model):.2f}%')

Base Model: Clean=88.77%, Robust(PGD-20)=0.40%
Sparsity: 0.00%


In [5]:
# ==========================================================================
# MAIN LOOP: Progressive ExPSO Pruning + Adversarial Fine-Tuning
# ==========================================================================
expert_ratios = [0.30, 0.50, 0.70]

for current_target in expert_ratios:
    print(f'\n{"="*70}')
    print(f'TRAINING EXPERT: {current_target*100:.0f}% Sparsity')
    print(f'{"="*70}')

    # Reload fresh model for each expert
    model = torchvision.models.resnet18(weights=None, num_classes=10).to(device)
    model = load_checkpoint(model, 'checkpoints/resnet18_standard_best.pth').float()

    # --- Algorithm 3: Progressive ExPSO Compression ---
    start_time = time.time()
    model = progressive_prune(
        model, trainloader,
        target_ratio=current_target,
        n_steps=PROGRESSIVE_STEPS,
        n_particles=EXPSO_PARTICLES,
        max_iter=EXPSO_ITER_PER_STEP,
        batch_size=32,
        device=str(device),
        use_adversarial_fitness=True,
        finetune_epochs=INTER_STEP_FT_EPOCHS,
        finetune_lr=0.01
    )
    prune_time = time.time() - start_time
    print(f'Progressive ExPSO time: {prune_time/60:.1f} minutes')

    # Evaluate after pruning (before fine-tuning)
    clean_pre = evaluate(model, testloader)
    robust_pre = evaluate(model, testloader,
        atk=torchattacks.PGD(model, eps=eps, alpha=alpha, steps=steps))
    sparsity_pre = count_zero_params(model)
    print(f'After pruning: Clean={clean_pre:.2f}%, Robust={robust_pre:.2f}%, Sparsity={sparsity_pre:.1f}%')

    # --- Adversarial Fine-Tuning (80 epochs, PGD-10) ---
    optimizer = optim.SGD(model.parameters(), lr=BASE_LR, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TRAIN_EPOCHS)

    best_rob = 0
    for epoch in range(1, TRAIN_EPOCHS + 1):
        train_one_epoch(model, trainloader, optimizer, nn.CrossEntropyLoss(), epoch,
            atk=torchattacks.PGD(model, eps=eps, alpha=alpha, steps=10))
        scheduler.step()

        if epoch % 20 == 0 or epoch == TRAIN_EPOCHS:
            rob = evaluate(model, testloader,
                atk=torchattacks.PGD(model, eps=eps, alpha=alpha, steps=20))
            if rob > best_rob:
                best_rob = rob
                fname = f'checkpoints/expso_expert_{int(round(current_target*100))}.pth'
                save_checkpoint(model, fname)
                print(f'--> Saved {current_target*100:.0f}% expert: {rob:.2f}% Robustness')

    del model, optimizer, scheduler
    gc.collect()
    torch.cuda.empty_cache()

print('\n--- ALL EXPERTS TRAINED WITH PROGRESSIVE COMPRESSION ---')


TRAINING EXPERT: 30% Sparsity

Progressive compression: 10% -> 20% -> 30%

Progressive Step 1/3: Pruning to 10.0%
Identifying prunable layers...
Found 16 prunable layers (Conv2d + Linear)
Initializing swarm with 30 particles...
Initial best fitness: 0.1334
Running ExPSO swarm optimization for TARGET: 10.0% Sparsity...


  3%|▎         | 1/30 [00:00<00:10,  2.84it/s]

Iter 0: Best fitness = -0.0199


 37%|███▋      | 11/30 [00:04<00:07,  2.67it/s]

Iter 10: Best fitness = -0.2192


...

--> Saved 70% expert: 34.76% Robustness


Epoch 60: 100%|██████████| 391/391 [01:15<00:00,  5.19it/s, loss=1.61, acc=38.8]


--> Saved 70% expert: 36.75% Robustness


Epoch 80: 100%|██████████| 391/391 [01:13<00:00,  5.31it/s, loss=1.53, acc=41.2]


--> Saved 70% expert: 37.37% Robustness

--- ALL EXPERTS TRAINED WITH PROGRESSIVE COMPRESSION ---


In [6]:
# ==========================================================================
# PHASE 2 FINAL RESULTS TABLE
# ==========================================================================
def evaluate_all(model, name):
    print(f'Evaluating {name}...')
    clean = evaluate(model, testloader)
    pgd20 = evaluate(model, testloader, atk=torchattacks.PGD(model, eps=eps, alpha=alpha, steps=20))
    pgd100 = evaluate(model, testloader, atk=torchattacks.PGD(model, eps=eps, alpha=alpha, steps=100))
    fgsm = evaluate(model, testloader, atk=torchattacks.FGSM(model, eps=eps))
    sparsity = count_zero_params(model)
    return {'Method': name, 'Clean': clean, 'PGD-20': pgd20, 'PGD-100': pgd100,
            'FGSM': fgsm, 'Sparsity': sparsity}

results = []
results.append({'Method': 'Base', 'Clean': clean_base, 'PGD-20': robust_base,
                'PGD-100': robust_base, 'FGSM': robust_base, 'Sparsity': 0})

# Load and evaluate magnitude baseline
if os.path.exists('checkpoints/mag_final_best.pth'):
    model_mag = torchvision.models.resnet18(weights=None, num_classes=10).to(device)
    model_mag = load_checkpoint(model_mag, 'checkpoints/mag_final_best.pth')
    results.append(evaluate_all(model_mag, 'Magnitude'))

# Evaluate all 3 ExPSO experts
for r in [30, 50, 70]:
    model_e = torchvision.models.resnet18(weights=None, num_classes=10).to(device)
    model_e = load_checkpoint(model_e, f'checkpoints/expso_expert_{r}.pth')
    results.append(evaluate_all(model_e, f'ExPSO-{r}'))

print(f'\n{"Method":<12} {"Clean":>8} {"PGD-20":>8} {"PGD-100":>8} {"FGSM":>8} {"Sparse":>8}')
print('-'*60)
for r in results:
    print(f'{r["Method"]:<12} {r["Clean"]:>7.2f}% {r["PGD-20"]:>7.2f}% {r["PGD-100"]:>7.2f}% {r["FGSM"]:>7.2f}% {r["Sparsity"]:>7.1f}%')

Evaluating Magnitude...
Evaluating ExPSO-30...
Evaluating ExPSO-50...
Evaluating ExPSO-70...

Method          Clean   PGD-20  PGD-100     FGSM   Sparse
------------------------------------------------------------
Base           88.77%    0.40%    0.40%    0.40%     0.0%
Magnitude      68.74%   38.86%   38.71%   43.72%    49.2%
ExPSO-30       71.45%   39.25%   39.06%   45.07%    30.3%
ExPSO-50       69.04%   38.91%   38.78%   44.13%    53.7%
ExPSO-70       65.98%   37.41%   37.23%   41.71%    73.7%
